In [1]:
# import libraries

import numpy as np
import pandas as pd
import re
import nltk
import string
from typing import List

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline

In [2]:
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\osaze\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\osaze\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
# load the datasets
df_train = pd.read_csv('../01-data/02-processed/train_clean.csv')
df_test = pd.read_csv('../01-data/02-processed/test_clean.csv')
df_val = pd.read_csv('../01-data/02-processed/val_clean.csv')

In [4]:
# load datasets
languages = {
    'hausa' : 'hau',
    'igbo': 'ibo',
    'pidgin': 'pcm',
    'yoruba': 'yor'
}


df_all = []

for lang, folder in languages.items():
    lang =  pd.read_csv(f'../01-data/03-stopwords/{folder}.csv')
    df_all.append(lang)

stopword_df = pd.concat(df_all, ignore_index=True)

stopword = set(stopword_df['word'].to_list())


In [5]:
lemmatizer = WordNetLemmatizer()

def dataset_preprocessing(text):
    # lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r'http\S+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map
    "\U0001F1E0-\U0001F1FF"  # flags
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE
    )

    text = emoji_pattern.sub('', text)

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [word for word in tokens if word not in stopword]

    # Lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

In [6]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36569 entries, 0 to 36568
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   tweet         36569 non-null  object
 1   label         36569 non-null  object
 2   language      36569 non-null  object
 3   tweet_length  36569 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 1.1+ MB


In [7]:
df_list = [df_train, df_test, df_val]
for df in df_list:
    df['cleaned_tweet'] = df['tweet'].apply(dataset_preprocessing)
    if df is df_train:
        df.drop(columns= ['tweet', "tweet_length"])
    else:
        df.drop(columns= ['tweet'])
    df.rename(columns={'cleaned_tweet': 'tweet'})

In [8]:
X_train = df_train['tweet']
y_train = df_train['label']
X_test = df_test['tweet']
y_test = df_test['label']
X_val = df_val['tweet']
y_val = df_val['label']

In [9]:
log_reg = Pipeline([
    ('vectorizer', TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=True)),
    ('model', LogisticRegression(max_iter=1000))
])

log_reg.fit(X_train, y_train)

predict = log_reg.predict(X_val)

accuracy = accuracy_score(y_val, predict)
print(accuracy)

0.7411501513357021


## Hyperparameter Tuning

In [13]:
def tune_base_model(features: list[int], X_train, y_train, X_val, y_val, **kwargs):
    
    scores = []

    for feature in features:
        log_reg = Pipeline([
        ('vectorizer', TfidfVectorizer(max_features=feature, 
                                       **kwargs)),
        ('model', LogisticRegression(max_iter=1000))
        ])

        log_reg.fit(X_train, y_train)

        predict = log_reg.predict(X_val)

        accuracy = accuracy_score(y_val, predict)
        
        scores.append((feature, accuracy))

    return pd.DataFrame(scores, columns = ['max_feature', 'accuracy'])

In [18]:
features = [10000, 20000, 50000, 60000, 70000]
ans = tune_base_model(features, X_train, y_train, X_val, y_val, ngram_range=(1,2), sublinear_tf=True)

In [19]:
ans

,max_feature,accuracy
0,10000,0.738913
1,20000,0.741150
2,50000,0.749836
3,60000,0.752994
4,70000,0.754310
